# 4. Context engineering

The loop from tutorial 3 will read whatever you stuff into the prompt. Stuffing is how a Tuesday huddle turns into context rot.

This notebook adds a note catalog. Titles go in the system prompt. Bodies come back through `read_note`. One note is over the cap so you can see the refusal. The picnic note is long and useless for a carton count. The demo should not load it.

Inventory and the policy tools are still here. The oat-milk count is checked against the shelf, not only against the note.

The cells below are the working agent, copied forward and extended. They include tool calling, shop data and files, and the agent loop. This notebook adds the next piece and still runs on its own.

## Setup

`DEMO_MODE = None` follows the environment variable of the same name. If that variable is unset, a missing `OPENAI_API_KEY` selects the scripted demo and a present key selects the Chat Completions API. The key is loaded from the repository-root `.env` by `labs/tutorial/common/client.py`. The value is not printed.

Companion notes: `4-context-engineering.md`.

In [ ]:
# None: use the API when OPENAI_API_KEY is set, otherwise the scripted demo.
# True: always the scripted demo. False: always the API (exits if the key is missing).
DEMO_MODE = None

import sys
from pathlib import Path


def find_root():
    """Repo root, whether the kernel started here or in labs/tutorial."""
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        marker = candidate / "labs" / "tutorial" / "common" / "client.py"
        if marker.is_file():
            return candidate
    raise RuntimeError(
        "Open this notebook inside the book-agents repository. "
        "The setup cell looks for labs/tutorial/common/client.py."
    )


ROOT = find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from labs.tutorial.runtime import chat, describe_mode, using_demo

LESSON = '4-context-engineering'
LESSON_NUMBER = 4
USING_DEMO = using_demo(DEMO_MODE)
print("tutorial:", LESSON)
print(describe_mode(DEMO_MODE))


## Shared helpers

Schema, message copying, and the demo check used by every later cell.

In [ ]:
import json
import re
import sqlite3
import time
from pathlib import Path

HERE = ROOT / "labs" / "tutorial"
DOCS = HERE / "docs"
DATA = HERE / "data"
VAR = HERE / "var"

HANDLERS = {}
TOOLS = []


def register(name, description, properties, required, fn):
    """Add one Chat Completions tool. ``fn`` runs only if the harness allows it."""
    spec = {
        "type": "function",
        "function": {
            "name": name,
            "description": description,
            "parameters": {
                "type": "object",
                "properties": properties,
                "required": required,
            },
        },
    }
    # Re-running a cell replaces the previous schema instead of stacking a second copy.
    TOOLS[:] = [tool for tool in TOOLS if tool["function"]["name"] != name]
    TOOLS.append(spec)
    HANDLERS[name] = fn


def preview(text, limit=500):
    """Shorten a tool result for the printed trace. The model still gets the full string."""
    text = text if isinstance(text, str) else str(text)
    if len(text) <= limit:
        return text
    return text[:limit] + "\n... [" + str(len(text)) + " characters]"


def assistant_message(turned):
    """Copy an assistant turn into the message list, including any tool calls."""
    message = {"role": "assistant", "content": turned["content"] or ""}
    calls = turned["tool_calls"] or []
    if calls:
        message["tool_calls"] = []
        for call in calls:
            raw = call.get("raw") or json.dumps(call["arguments"], sort_keys=True)
            message["tool_calls"].append({
                "id": call["id"],
                "type": "function",
                "function": {"name": call["name"], "arguments": raw},
            })
    return message


def check(condition, message):
    """Fail the demo when the scripted path drifts. Warn, and continue, on a live model."""
    if condition:
        print("check ok:", message)
        return
    if USING_DEMO:
        raise AssertionError(message)
    print("check warning (live model):", message)


def base_rules():
    return (
        "You are the counter concierge for Hearth Lane Café. "
        "Shop facts come from tools, not from guesses in the prompt. "
        "If a tool does not say, say you don't know. "
        "Cite the tool or the docs path you used. "
        "Do not invent a Wi-Fi password, a refund, or a shipping exception. "
        "Do not charge a card or send email yourself."
    )


def call_tool(call):
    """Run one handler. Bad arguments become an ERROR string, not a crash."""
    name = call["name"]
    args = call["arguments"]
    if not isinstance(args, dict):
        return "ERROR: arguments must be a JSON object."
    if args.get("_error"):
        return "ERROR: " + str(args["_error"])
    fn = HANDLERS.get(name)
    if fn is None:
        return "ERROR: unknown tool " + repr(name) + "."
    return fn(args)


## Tutorial 1 — one tool

`get_shop_fact` opens the shop document. The model only sees the string you return.

In [ ]:
from labs.tutorial.common.tools import read_file as read_docs_file


def _read_docs(path):
    """Read one shop document. The path jail lives in ``labs.tutorial.common.tools``."""
    if not isinstance(path, str):
        return "ERROR: path must be a string. Example: policy.md"
    return read_docs_file(str(DOCS), path)


def _section(markdown, heading):
    lines = markdown.splitlines()
    start = None
    prefix = "## " + heading
    for index, line in enumerate(lines):
        if line.strip() == prefix:
            start = index + 1
            break
    if start is None:
        return ""
    body = []
    for line in lines[start:]:
        if line.startswith("## "):
            break
        body.append(line)
    return "\n".join(body).strip()


def get_shop_fact(args):
    """Return one policy or FAQ section. The model names a topic; this function opens the file."""
    topic = str(args.get("topic", "")).strip().lower()
    mapping = {
        "returns": ("policy.md", "Returns"),
        "shipping": ("policy.md", "Shipping"),
        "hours": ("faq.md", "Where and when"),
        "allergens": ("faq.md", "Allergens"),
    }
    if topic not in mapping:
        return "ERROR: topic must be returns, shipping, hours, or allergens."
    path, heading = mapping[topic]
    document = _read_docs(path)
    if document.startswith("ERROR:"):
        return document
    body = _section(document, heading)
    if not body:
        return "ERROR: " + heading + " is missing from " + path + "."
    return "TOPIC: " + topic + "\nSOURCE: docs/" + path + "\n\n" + body


register(
    "get_shop_fact",
    "Look up one Hearth Lane rule. topic is returns, shipping, hours, or allergens. "
    "The result quotes docs/policy.md or docs/faq.md.",
    {"topic": {"type": "string", "description": "returns, shipping, hours, or allergens"}},
    ["topic"],
    get_shop_fact,
)


## Tutorial 2 — files and the shelf

`read_shop_file` uses the path jail in `labs/tutorial/common/tools.py`. `query_inventory` reads the shelf seeded in this notebook. The SQL stays in the function.

In [ ]:
# Shelf seed for this tutorial. The model never writes the SQL.
# sku, name, category, unit, reorder_point, on_hand, par.
SEED_ROWS = (
    ("HB-12", "House blend 12oz", "coffee", "bag", 6, 4, 18),
    ("HB-2LB", "House blend 2lb", "coffee", "bag", 4, 9, 10),
    ("ES-1KG", "Espresso beans 1kg", "coffee", "bag", 5, 5, 12),
    ("OM-32", "Oat milk 32oz", "dairy", "carton", 8, 3, 16),
    ("MLK-1", "Whole milk gallon", "dairy", "gallon", 4, 11, 12),
    ("ALM-1", "Almond meal", "bakery", "bag", 2, 1, 4),
    ("FL-01", "Paper filters", "supply", "box", 2, 7, 8),
)

STATE = {"conn": None}


def gap_for(row):
    """Cartons or bags to buy to reach par. The supplier note does not get a vote."""
    return int(row["par"]) - int(row["on_hand"])


def reset_db():
    if STATE["conn"] is not None:
        STATE["conn"].close()
    conn = sqlite3.connect(":memory:")
    conn.row_factory = sqlite3.Row
    conn.executescript(
        """
        CREATE TABLE products (
            sku TEXT PRIMARY KEY,
            name TEXT NOT NULL,
            category TEXT NOT NULL,
            unit TEXT NOT NULL,
            reorder_point INTEGER NOT NULL
        );
        CREATE TABLE inventory (
            sku TEXT PRIMARY KEY,
            on_hand INTEGER NOT NULL,
            par INTEGER NOT NULL
        );
        """
    )
    for sku, name, category, unit, reorder_point, on_hand, par in SEED_ROWS:
        conn.execute(
            "INSERT INTO products (sku, name, category, unit, reorder_point) VALUES (?, ?, ?, ?, ?)",
            (sku, name, category, unit, reorder_point),
        )
        conn.execute(
            "INSERT INTO inventory (sku, on_hand, par) VALUES (?, ?, ?)",
            (sku, on_hand, par),
        )
    conn.commit()
    STATE["conn"] = conn


def row_for(sku):
    if STATE["conn"] is None:
        reset_db()
    found = STATE["conn"].execute(
        """
        SELECT p.sku, p.name, p.category, p.unit, p.reorder_point, i.on_hand, i.par
        FROM products p
        JOIN inventory i ON p.sku = i.sku
        WHERE p.sku = ?
        """,
        (sku,),
    ).fetchone()
    if found is None:
        return None
    item = dict(found)
    item["gap"] = gap_for(item)
    return item


def query_inventory(args):
    """Read the shelf. The SQL stays in this function. The model passes filters, not a statement."""
    if STATE["conn"] is None:
        reset_db()
    only_low = bool(args.get("only_low", False))
    sku = args.get("sku")
    sql = (
        "SELECT p.sku, p.name, p.category, p.unit, p.reorder_point, i.on_hand, i.par "
        "FROM products p JOIN inventory i ON p.sku = i.sku"
    )
    params = []
    where = []
    if isinstance(sku, str) and sku.strip():
        where.append("p.sku = ?")
        params.append(sku.strip())
    if only_low:
        where.append("i.on_hand <= p.reorder_point")
    if where:
        sql += " WHERE " + " AND ".join(where)
    sql += " ORDER BY p.sku"
    rows = []
    for found in STATE["conn"].execute(sql, params):
        item = dict(found)
        item["gap"] = gap_for(item)
        rows.append(item)
    payload = {"rows": rows}
    if isinstance(sku, str) and sku.strip() and not rows:
        payload["error"] = "unknown sku " + sku.strip()
    return json.dumps(payload)


def read_shop_file(args):
    """Model-facing file tool. Only docs/policy.md and docs/faq.md resolve."""
    return _read_docs(args.get("path", ""))


register(
    "read_shop_file",
    "Read a shop document. path is policy.md or faq.md. "
    "policy.md has returns, shipping, damage, and local delivery. "
    "faq.md has hours, the menu, and allergens. "
    "Paths outside docs/ return ERROR.",
    {"path": {"type": "string", "description": "policy.md or faq.md"}},
    ["path"],
    read_shop_file,
)
register(
    "query_inventory",
    "Read the café shelf from SQLite. Low stock means on_hand <= reorder_point. "
    "Each row includes gap, which is par minus on_hand. This tool does not write.",
    {
        "only_low": {"type": "boolean", "description": "True to keep only low or at-reorder rows."},
        "sku": {"type": "string", "description": "Optional sku, such as OM-32."},
    },
    [],
    query_inventory,
)


## Tutorial 4 — notes under a budget

The prompt gets titles. `read_note` returns one short body. The picnic note is over the cap on purpose.

In [ ]:
NOTES = DATA / "notes"
PER_NOTE_CAP = 500
TURN_BUDGET = 600


def _note_path(name):
    if not isinstance(name, str) or not name.strip():
        return "ERROR: name is required. Example: oat-milk.md"
    cleaned = name.strip().replace("\\", "/")
    if "/" in cleaned or cleaned.startswith("."):
        return "ERROR: name must be a file in the notes catalog, not a path."
    if not cleaned.endswith(".md"):
        return "ERROR: notes are markdown files, such as oat-milk.md."
    candidate = (NOTES / cleaned).resolve()
    if candidate.parent != NOTES.resolve():
        return "ERROR: name escapes the notes folder."
    if not candidate.is_file():
        return "ERROR: " + cleaned + " is not in the catalog."
    return candidate


def _note_title(text):
    for line in text.splitlines():
        if line.startswith("# "):
            return line[2:].strip()
    return "(untitled)"


def list_notes(args):
    """Titles and sizes only. Bodies stay on disk until read_note."""
    del args
    lines = ["NOTES"]
    for path in sorted(NOTES.glob("*.md")):
        text = path.read_text(encoding="utf-8")
        lines.append(
            "- " + path.name + " | " + _note_title(text) + " | " + str(len(text)) + " chars"
        )
    return "\n".join(lines)


def read_note(args):
    found = _note_path(args.get("name", ""))
    if isinstance(found, str):
        return found
    text = found.read_text(encoding="utf-8")
    if len(text) > PER_NOTE_CAP:
        return (
            "ERROR: " + found.name + " is " + str(len(text))
            + " characters. The per-note cap is " + str(PER_NOTE_CAP)
            + ". File a shorter note."
        )
    return "NOTE: " + found.name + "\n\n" + text


def paste_chars():
    total = 0
    for path in NOTES.glob("*.md"):
        total += len(path.read_text(encoding="utf-8"))
    return total


register(
    "list_notes",
    "List huddle notes by name, title, and size. Does not return bodies.",
    {},
    [],
    list_notes,
)
register(
    "read_note",
    "Read one short note by file name, such as oat-milk.md. "
    "A note over the per-note cap returns ERROR. The picnic note is an example.",
    {"name": {"type": "string", "description": "File name from list_notes."}},
    ["name"],
    read_note,
)


## System prompt

The prompt stays short. Documents, notes, skills, and pages arrive through tools.

In [ ]:
def system_text():
    parts = [base_rules()]
    parts.append("Use get_shop_fact before you state a return, shipping, hours, or allergen rule.")
    parts.append("Use read_shop_file for policy.md and faq.md, and query_inventory for the shelf. Do not invent stock counts.")
    parts.append("Note catalog (titles only, not bodies):\n" + list_notes({}))
    parts.append("Read a note before you quote it. Skip notes that are not about the question. A note over the cap returns ERROR.")
    return "\n\n".join(parts)


## Agent loop

One step is one model call. Tool results are the observation. The loop stops on a final answer, on `max_steps`, or on a repeated call. It does not write a customer answer when it stops early.

In [ ]:
MAX_IDENTICAL_CALLS = 2
REPEAT_NOTE = (
    "\n\nNOTE: You already made this call. Answer the question without repeating it."
)

def run_agent(user_text, lesson, system, max_steps=6):
    """Perceive the thread, let the model reason, act on tool calls, observe the results."""
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user_text},
    ]
    seen = {}
    log = []
    spans = []
    usage_rows = []
    for step in range(1, max_steps + 1):
        turned = chat(messages, TOOLS, lesson=lesson, demo=DEMO_MODE)
        messages.append(assistant_message(turned))
        calls = turned["tool_calls"] or []
        if not calls:
            text = (turned["content"] or "").strip() or "(empty answer)"
            print("stop: final after " + str(step) + " model call(s)")
            return _finish(text, step, "final", log, spans, usage_rows)
        for call in calls:
            signature = call["name"] + " " + json.dumps(call["arguments"], sort_keys=True, default=str)
            count = seen.get(signature, 0) + 1
            seen[signature] = count
            if count > MAX_IDENTICAL_CALLS:
                log.append("step " + str(step) + ": repeated " + signature)
                print("stop: repeated_call after " + str(step) + " model call(s)")
                text = (
                    "Stopped: the model repeated the same tool call ("
                    + call["name"] + ") more than " + str(MAX_IDENTICAL_CALLS)
                    + " times. The harness ended the loop."
                )
                return _finish(text, step, "repeated_call", log, spans, usage_rows)
            result = _invoke(call)
            if count == MAX_IDENTICAL_CALLS:
                result += REPEAT_NOTE
            first = result.splitlines()[0] if result else "(empty)"
            log.append(
                "step " + str(step) + ": " + call["name"] + " "
                + json.dumps(call["arguments"], sort_keys=True, default=str)
                + " -> " + first
            )
            messages.append({
                "role": "tool",
                "tool_call_id": call["id"],
                "content": result,
            })
            print("[step " + str(step) + "] " + call["name"] + " " + json.dumps(call["arguments"], sort_keys=True))
            print(preview(result))
            print()
    text = "Stopped: reached max steps (" + str(max_steps) + ") without a final answer. The harness did not write one."
    print("stop: max_steps after " + str(max_steps) + " model call(s)")
    return _finish(text, max_steps, "max_steps", log, spans, usage_rows)


def _invoke(call):
    return call_tool(call)


def _finish(text, steps, stopped, log, spans, usage_rows):
    return {
        "text": text,
        "steps": steps,
        "stopped": stopped,
        "tool_log": log,
        "spans": spans,
        "usage": usage_rows,
    }


## Demo

Run this cell after the ones above. In demo mode the checks raise if the trace drifts. A live model prints a warning instead, because the wording can vary.

In [ ]:
reset_db()
catalog = list_notes({})
print("paste_all_chars:", paste_chars())
print("catalog_chars:", len(catalog))
print("TURN_BUDGET:", TURN_BUDGET)
print("PER_NOTE_CAP:", PER_NOTE_CAP)
print(catalog)
print()
print("oversized note:")
print(read_note({"name": "picnic.md"}))
print()
check(paste_chars() > TURN_BUDGET, "pasting every note exceeds the turn budget")
check(len(catalog) < TURN_BUDGET, "the title catalog fits in the budget")

question = (
    "From the huddle notes, how many oat milk cartons were on hand, "
    "and how many cardamom buns did we bake on Thursday? Check the shelf for OM-32. "
    "Do not load the picnic note."
)
result = run_agent(question, "4-context", system_text())
print("ANSWER:", result["text"])
joined = "\n".join(result["tool_log"])
check(result["stopped"] == "final", "context demo finished")
check("oat-milk.md" in joined and "thursday-buns.md" in joined, "the relevant notes were read")
check("picnic.md" not in joined, "the picnic note was not loaded into the loop")
check("3" in result["text"] and "24" in result["text"], "both counts are in the answer")
check("query_inventory" in joined, "the shelf tool from tutorial 2 still runs")
